In [1]:
from pathlib import Path

BASE = Path(r"E:\AI_Detect")

folders = [

    "data/image/human/raw/wikimedia",
    "data/image/human/raw/nasa",
    "data/image/human/raw/loc",
    "data/image/human/raw/archive",

    "data/image/human/metadata"
]

for f in folders:

    (BASE / f).mkdir(
        parents=True,
        exist_ok=True
    )

print("Folders created.")

Folders created.


In [11]:
WIKIMEDIA_TARGET = 1000
NASA_TARGET = 1000
LOC_TARGET = 1000
ARCHIVE_TARGET = 1000
WIKIMEDIA_TARGET = 5000

In [3]:
import pandas as pd
import requests

from pathlib import Path
from tqdm import tqdm

BASE = Path(r"E:\AI_Detect")

META_FILE = (
    BASE /
    "data/image/human/metadata/image_metadata.csv"
)

if META_FILE.exists():

    meta_df = pd.read_csv(META_FILE)

else:

    meta_df = pd.DataFrame()

existing_files = set(
    meta_df.filename.tolist()
) if len(meta_df) else set()

In [12]:
NASA_TARGET = 1000

save_dir = (
    BASE /
    "data/image/human/raw/nasa"
)

saved = 0

page = 1

pbar = tqdm(total=NASA_TARGET)

while saved < NASA_TARGET:

    url = (
        "https://images-api.nasa.gov/search"
    )

    params = {

        "media_type": "image",

        "page": page

    }

    try:

        r = requests.get(
            url,
            params=params,
            timeout=30
        )

        items = (
            r.json()
            ["collection"]
            ["items"]
        )

        if len(items) == 0:
            break

        for item in items:

            try:

                img_url = (
                    item["links"][0]["href"]
                )

                import hashlib

                fname = hashlib.md5(img_url.encode()).hexdigest() + ".jpg"

                if fname in existing_files:
                    continue

                img = requests.get(
                    img_url,
                    timeout=30
                )

                with open(
                    save_dir / fname,
                    "wb"
                ) as f:

                    f.write(img.content)

                existing_files.add(fname)

                meta_df = pd.concat([

                    meta_df,

                    pd.DataFrame([{

                        "filename": fname,

                        "source": "nasa",

                        "url": img_url

                    }])

                ])

                saved += 1

                pbar.update(1)

                if saved >= NASA_TARGET:
                    break

            except:
                pass

        page += 1

    except:
        break

pbar.close()



100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [10:35<00:00,  1.57it/s]


In [13]:


# ====================================================
# SAVE METADATA
# ====================================================

meta = pd.DataFrame(metadata_rows)

meta.to_csv(
    META_FILE,
    index=False
)

print()
print("Downloaded:", saved)
print("Metadata:", META_FILE)


save_dir = (
    BASE /
    "data/image/human/raw/loc"
)

saved = 0

offset = 0

pbar = tqdm(total=LOC_TARGET)

while saved < LOC_TARGET:

    url = (
        "https://www.loc.gov/photos/"
    )

    params = {

        "fo": "json",

        "c": 100,

        "sp": offset

    }

    try:

        r = requests.get(
            url,
            params=params,
            timeout=30
        )

        data = r.json()

        results = data.get(
            "results",
            []
        )

        if len(results) == 0:
            break

        for item in results:

            try:

                img_url = item["image_url"][0]

                fname = img_url.split("/")[-1]

                if fname in existing_files:
                    continue

                img = requests.get(
                    img_url,
                    timeout=30
                )

                with open(
                    save_dir / fname,
                    "wb"
                ) as f:

                    f.write(img.content)

                existing_files.add(fname)

                meta_df = pd.concat([

                    meta_df,

                    pd.DataFrame([{

                        "filename": fname,

                        "source": "loc",

                        "url": img_url

                    }])

                ])

                saved += 1

                pbar.update(1)

                if saved >= LOC_TARGET:
                    break

            except:
                pass

        offset += 1

    except:
        break

pbar.close()

save_dir = (
    BASE /
    "data/image/human/raw/archive"
)

saved = 0

page = 1

pbar = tqdm(total=ARCHIVE_TARGET)

while saved < ARCHIVE_TARGET:

    params = {

        "q": (
            "mediatype:image "
            "AND year:[2000 TO 2021]"
        ),

        "rows": 100,

        "page": page,

        "output": "json"

    }

    try:

        r = requests.get(

            "https://archive.org/advancedsearch.php",

            params=params,

            timeout=30

        )

        docs = (
            r.json()
            ["response"]
            ["docs"]
        )

        if len(docs) == 0:
            break

        for doc in docs:

            try:

                identifier = doc["identifier"]

                meta_url = (
                    f"https://archive.org/metadata/"
                    f"{identifier}"
                )

                meta = requests.get(
                    meta_url
                ).json()

                files = meta.get(
                    "files",
                    []
                )

                for f in files:

                    name = f["name"]

                    if not (
                        name.endswith(".jpg")
                        or
                        name.endswith(".jpeg")
                        or
                        name.endswith(".png")
                    ):
                        continue

                    url = (
                        f"https://archive.org/download/"
                        f"{identifier}/{name}"
                    )

                    fname = (
                        identifier +
                        "_" +
                        name
                    )

                    if fname in existing_files:
                        continue

                    img = requests.get(
                        url,
                        timeout=30
                    )

                    with open(
                        save_dir / fname,
                        "wb"
                    ) as out:

                        out.write(
                            img.content
                        )

                    existing_files.add(
                        fname
                    )

                    meta_df = pd.concat([

                        meta_df,

                        pd.DataFrame([{

                            "filename": fname,

                            "source": "archive",

                            "url": url

                        }])

                    ])

                    saved += 1

                    pbar.update(1)

                    if saved >= ARCHIVE_TARGET:
                        break

            except:
                pass

        page += 1

    except:
        break

pbar.close()

meta_df.to_csv(
    META_FILE,
    index=False
)

print()
print("Total Images:",
      len(meta_df))

print("Metadata Saved:")
print(META_FILE)


Downloaded: 1000
Metadata: E:\AI_Detect\data\image\human\metadata\wikimedia_metadata.csv


  1%|▉                                                                             | 12/1000 [01:29<2:02:08,  7.42s/it]
1050it [50:09,  2.87s/it]                                                                                              


Total Images: 4146
Metadata Saved:
E:\AI_Detect\data\image\human\metadata\wikimedia_metadata.csv



Downloaded: 1
Metadata: E:\AI_Detect\data\image\human\metadata\wikimedia_metadata.csv


100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [34:18<00:00,  2.06s/it]
1083it [56:25,  3.13s/it]                                                                                              


Total Images: 2084
Metadata Saved:
E:\AI_Detect\data\image\human\metadata\wikimedia_metadata.csv
